In [ ]:
!pip install numpy==1.23.5
!pip install gensim
!pip install jax==0.4.13
!pip install nltk

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

from tensorflow.keras.callbacks import EarlyStopping

import gensim.downloader as api

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
df = pd.read_csv('/content/Book_review.csv')

In [ ]:
df.head()

In [ ]:
print(df.shape)
print(df.columns)

In [ ]:
df = df[['review/text', 'review/score']]

In [ ]:
df.columns = ['review', 'rating']

In [ ]:
df = df[df['rating'] != 3]

In [ ]:
df['sentiment'] = df['rating'].apply(lambda x: 1 if x >= 4 else 0)

In [ ]:
df['sentiment'].value_counts()

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'http\S+', '', text)

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()

    cleaned_words = []

    for word in words:

        if word not in stop_words:

            word = lemmatizer.lemmatize(word)

            cleaned_words.append(word)

    return ' '.join(cleaned_words)

In [ ]:
df['clean_review'] = df['review'].apply(clean_text)

In [ ]:
df[['review', 'clean_review']].head()

In [ ]:
max_words = 10000

tokenizer = Tokenizer(num_words=max_words)

tokenizer.fit_on_texts(df['clean_review'])

sequences = tokenizer.texts_to_sequences(df['clean_review'])

In [ ]:
sequence_lengths = [len(seq) for seq in sequences]

max_len = int(np.percentile(sequence_lengths, 95))

padded_sequences = pad_sequences(
    sequences,
    maxlen=max_len,
    padding='post'
)

print(max_len)

In [ ]:
X = padded_sequences
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
model_rnn = Sequential()

model_rnn.add(
    Embedding(
        input_dim=max_words,
        output_dim=128,
        input_length=max_len
    )
)

model_rnn.add(SimpleRNN(64))

model_rnn.add(Dropout(0.5))

model_rnn.add(Dense(1, activation='sigmoid'))

model_rnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_rnn.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history_rnn = model_rnn.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    callbacks=[early_stop]
)

In [ ]:
rnn_pred = model_rnn.predict(X_test)

rnn_pred = (rnn_pred > 0.5).astype(int)

print(accuracy_score(y_test, rnn_pred))

print(classification_report(y_test, rnn_pred))

In [ ]:
model_lstm = Sequential()

model_lstm.add(
    Embedding(
        input_dim=max_words,
        output_dim=128,
        input_length=max_len
    )
)

model_lstm.add(LSTM(64))

model_lstm.add(Dropout(0.5))

model_lstm.add(Dense(1, activation='sigmoid'))

model_lstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_lstm.summary()

In [ ]:
history_lstm = model_lstm.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    callbacks=[early_stop]
)

In [ ]:
lstm_pred = model_lstm.predict(X_test)

lstm_pred = (lstm_pred > 0.5).astype(int)

print(accuracy_score(y_test, lstm_pred))

print(classification_report(y_test, lstm_pred))

In [ ]:
embedding_model = api.load('glove-wiki-gigaword-50')

In [ ]:
word_index = tokenizer.word_index

vocab_size = min(max_words, len(word_index) + 1)

embedding_dim = 50

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in word_index.items():

    if i < vocab_size:

        if word in embedding_model:

            embedding_matrix[i] = embedding_model[word]

In [ ]:
model_w2v = Sequential()

model_w2v.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=False
    )
)

model_w2v.add(LSTM(64))

model_w2v.add(Dropout(0.5))

model_w2v.add(Dense(1, activation='sigmoid'))

model_w2v.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_w2v.summary()

In [ ]:
history_w2v = model_w2v.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    callbacks=[early_stop]
)

In [ ]:
w2v_pred = model_w2v.predict(X_test)

w2v_pred = (w2v_pred > 0.5).astype(int)

print(accuracy_score(y_test, w2v_pred))

print(classification_report(y_test, w2v_pred))

In [ ]:
def predict_sentiment(text):

    text = clean_text(text)

    seq = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(
        seq,
        maxlen=max_len,
        padding='post'
    )

    prediction = model_w2v.predict(padded)[0][0]

    if prediction >= 0.5:

        print('Positive Review')

    else:

        print('Negative Review')

predict_sentiment('This book was absolutely amazing and emotional')